# Faster R-CNN: training and test submissions on Kaggle

Run All currently restores the three completed data experiments and exports test CSVs
on the Kaggle GPU without training. We retain verified CSVs and checkpoint identities
on Google Drive. Enable Internet, attach the competition dataset and enable
`HM2_DRIVE_TOKEN` access. Use the updated `controlled_data_submissions_project.zip` code bundle.
For future training, enable `SHOULD_TRAIN`; submissions are generated after each completed run.

In [ ]:
import hashlib
import importlib
import importlib.util
import json
import logging
import math
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from zipfile import ZipFile

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12 or newer runtime.")

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()
requirements = {
    "polars": "polars>=1.31",
    "plotly": "plotly>=6",
    "pycocotools": "pycocotools>=2.0.11",
    "decouple": "python-decouple>=3.8",
    "PIL": "Pillow>=11",
    "tqdm": "tqdm>=4.67",
    "ipywidgets": "ipywidgets>=8",
    "googleapiclient": "google-api-python-client",
    "google.oauth2": "google-auth",
}
missing = [
    package for module, package in requirements.items() if importlib.util.find_spec(module) is None
]
if IS_KAGGLE:
    uv = shutil.which("uv")
    if uv is None:
        installer = Path("/kaggle/working/install_uv.sh")
        urllib.request.urlretrieve("https://astral.sh/uv/install.sh", installer)
        subprocess.run(["sh", str(installer)], check=True)
        uv = str(Path.home() / ".local/bin/uv")
    if missing:
        subprocess.run(
            [uv, "pip", "install", "--system", "--python", sys.executable, *missing], check=True
        )

import polars as pl
import torch
import torchvision

from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
LOGGER = logging.getLogger("data_experiments_kaggle")
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
pl.from_dicts(
    [
        {
            "Python": platform.python_version(),
            "PyTorch": torch.__version__,
            "TorchVision": torchvision.__version__,
            "CUDA": torch.version.cuda,
            "GPU": torch.cuda.get_device_name() if torch.cuda.is_available() else "None",
            "Free VRAM GiB": (
                torch.cuda.mem_get_info()[0] / 1024**3 if torch.cuda.is_available() else 0.0
            ),
        }
    ]
)

## 1. Code and experiment settings

We use three independent Faster R-CNN configurations. Each starts from the original
pretrained weights unless resuming its own checkpoint. The session budget applies
to the entire queue, and all three runs use the same stride-five selection and validation.

In [ ]:
if IS_KAGGLE:
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload

    UserSecretsClient = importlib.import_module("kaggle_secrets").UserSecretsClient
    token = json.loads(UserSecretsClient().get_secret("HM2_DRIVE_TOKEN"))
    if not token.get("refresh_token"):
        raise ValueError("HM2_DRIVE_TOKEN must contain refreshable Drive credentials.")
    credentials = Credentials.from_authorized_user_info(token)
    del token
    service = build("drive", "v3", credentials=credentials, cache_discovery=False)
    drive_root = "16wH9gNp13kfa4T039nsiFcZwYOlEJI5Z"
    root_files = (
        service.files()
        .list(
            q=f"'{drive_root}' in parents and trashed = false",
            fields="files(id,name)",
            pageSize=1000,
        )
        .execute(num_retries=3)["files"]
    )
    inputs = [entry for entry in root_files if entry["name"] == "inputs"]
    if len(inputs) != 1:
        raise ValueError("Expected one inputs folder in hm2_retinanet on Drive.")
    input_files = (
        service.files()
        .list(
            q=f"'{inputs[0]['id']}' in parents and trashed = false",
            fields="files(id,name,size,md5Checksum)",
            pageSize=1000,
        )
        .execute(num_retries=3)["files"]
    )
    projects = [
        entry for entry in input_files if entry["name"] == "controlled_data_submissions_project.zip"
    ]
    if len(projects) != 1:
        raise FileNotFoundError(
            "Upload exactly one controlled_data_submissions_project.zip to hm2_retinanet/inputs on Drive."
        )
    project = projects[0]
    archive_path = Path("/kaggle/temp/controlled_data_submissions_project.zip")
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive_path.with_suffix(".incoming")
    with temporary.open("wb") as stream:
        transfer = MediaIoBaseDownload(stream, service.files().get_media(fileId=project["id"]))
        has_completed = False
        while not has_completed:
            _, has_completed = transfer.next_chunk(num_retries=3)
    with temporary.open("rb") as stream:
        actual_md5 = hashlib.file_digest(stream, "md5").hexdigest()
    if actual_md5 != project["md5Checksum"] or temporary.stat().st_size != int(project["size"]):
        raise ValueError("Project archive transfer checksum mismatch.")
    temporary.replace(archive_path)
    REPOSITORY_ROOT = Path("/kaggle/temp/hm2_data_experiments")
    REPOSITORY_ROOT.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive_path) as archive:
        required_paths = {
            "src/hm2/faster_rcnn.py",
            "src/hm2/faster_rcnn_experiments.py",
            "src/hm2/drive_api.py",
            "config/faster_rcnn_baseline.toml",
            "config/data_photometric_augmentation.toml",
            "config/data_input_resolution.toml",
            "config/yolo_baseline.toml",
            "config/data_repeat_factor_sampling.toml",
            "experiments/protocol.py",
            "experiments/yolo_baseline.py",
            "experiments/photometric_augmentation.py",
            "experiments/input_resolution.py",
            "experiments/repeat_factor_sampling.py",
            "data/metadata/frames.parquet",
            "data/metadata/objects.parquet",
            "data/metadata/ignored_regions.parquet",
        }
        if not required_paths.issubset(archive.namelist()):
            raise ValueError("The Drive archive is missing Faster R-CNN code or configuration.")
        for entry in archive.infolist():
            if (
                not (REPOSITORY_ROOT / entry.filename)
                .resolve()
                .is_relative_to(REPOSITORY_ROOT.resolve())
            ):
                raise ValueError("Project archive contains an unsafe path.")
        archive.extractall(REPOSITORY_ROOT)
    LOGGER.info("Downloaded and verified project code from Drive: %s", archive_path.name)
else:
    working_directory = Path.cwd()
    REPOSITORY_ROOT = (
        working_directory.parent
        if working_directory.name in {"notebooks", "experiments"}
        else working_directory
    )
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

sys.path.insert(0, str(REPOSITORY_ROOT))

from experiments.protocol import (
    BATCH_ID,
    export_submission,
    load_records,
    load_settings,
    metric_table,
    train_experiment,
)
from hm2.detection_data import select_frames
from hm2.experiment_checkpoint import publish_snapshot, write_json
from hm2.faster_rcnn_experiments import (
    build_experiment_config,
    compute_source_identity,
    find_competition_root,
    stage_images,
    verify_selection,
)

EXPERIMENTS = (
    "photometric_augmentation",
    "repeat_factor_sampling",
    "input_resolution",
)
SHOULD_TRAIN = False
SHOULD_EXPORT_SUBMISSIONS = True
SESSION_HOURS = 10.0
SETTINGS_BY_EXPERIMENT = {name: load_settings(REPOSITORY_ROOT, name) for name in EXPERIMENTS}
METADATA = REPOSITORY_ROOT / "data/metadata"
LOCAL_BATCH = Path("/kaggle/working/runs/data_experiments") / BATCH_ID
SUBMISSIONS_ROOT = Path("/kaggle/working/submissions")
OUTPUTS = {name: LOCAL_BATCH / name for name in EXPERIMENTS}
for output in OUTPUTS.values():
    output.mkdir(parents=True, exist_ok=True)
LOGGER.info(
    "Execution settings: experiments=%s, training=%s, export=%s, total_session_hours=%s",
    ", ".join(EXPERIMENTS),
    SHOULD_TRAIN,
    SHOULD_EXPORT_SUBMISSIONS,
    SESSION_HOURS,
)
pl.from_dicts(
    [
        {
            "Order": index + 1,
            "Experiment": name,
            "Resize": f"{settings['model']['min_size']}/{settings['model']['max_size']}",
            "Epochs": settings["training"]["epochs"],
            "Batch size": settings["training"]["batch_size"],
            "Learning rate": settings["training"]["learning_rate"],
            "Drive outputs": f"hm2_retinanet/batches/{BATCH_ID}/{name}",
        }
        for index, name in enumerate(EXPERIMENTS)
        for settings in [SETTINGS_BY_EXPERIMENT[name]]
    ]
)

## 2. Dataset staging

Inference-only stages the test images and original test_images.csv. When training is
enabled we also stage the selected training frames and full validation. Annotation files
remain byte-identical so checkpoint recipe verification uses the same source identity.

In [ ]:
SOURCE_ROOT = find_competition_root(KAGGLE_INPUT)
frames, objects, ignored = (
    pl.read_parquet(METADATA / f"{name}.parquet")
    for name in ("frames", "objects", "ignored_regions")
)
selected, validation = select_frames(frames, "train", 5), select_frames(frames, "val")
verify_selection(
    selected,
    validation,
    {
        "expected_selected_frames": 4868,
        "expected_selected_clips": 56,
        "expected_validation_frames": 2846,
    },
)
IMAGE_ROOT = SOURCE_ROOT
if SHOULD_TRAIN or SHOULD_EXPORT_SUBMISSIONS:
    if not torch.cuda.is_available():
        raise RuntimeError("Explicit GPU execution requires a CUDA runtime.")
    staged = Path("/kaggle/temp/data_experiment_images")
    needed_paths = []
    if SHOULD_TRAIN:
        needed_paths.extend(pl.concat([selected, validation])["relative_path"].to_list())
    if SHOULD_EXPORT_SUBMISSIONS:
        test = pl.read_csv(
            SOURCE_ROOT / "test_images.csv", schema_overrides={"image_id": pl.String}
        )
        if test.height != frames.filter(pl.col("split") == "test").height:
            raise ValueError("Test metadata coverage differs from the prepared dataset.")
        needed_paths.extend(test["file_name"].to_list())
    expected_bytes = sum((SOURCE_ROOT / path).stat().st_size for path in needed_paths)
    if shutil.disk_usage(LOCAL_BATCH).free < expected_bytes + 4 * 1024**3:
        raise OSError("Insufficient local disk for images, weights and checkpoints.")
    display(stage_images(SOURCE_ROOT, staged, needed_paths))
    for split in ("train", "val"):
        destination = staged / split / "annotations_coco.json"
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(SOURCE_ROOT / split / "annotations_coco.json", destination)
    if SHOULD_EXPORT_SUBMISSIONS:
        shutil.copyfile(SOURCE_ROOT / "test_images.csv", staged / "test_images.csv")
        if (SOURCE_ROOT / "sample_submission.csv").is_file():
            shutil.copyfile(SOURCE_ROOT / "sample_submission.csv", staged / "sample_submission.csv")
    IMAGE_ROOT = staged
    if SHOULD_TRAIN:
        from hm2.faster_rcnn import IgnoreAwareDetectionDataset

        for split, rows in (("train", selected), ("val", validation)):
            image, target = IgnoreAwareDetectionDataset(IMAGE_ROOT, rows.head(1), objects, ignored)[
                0
            ]
            if list(image.shape[-2:]) != target["original_size"].tolist():
                raise ValueError(f"Image dimensions differ from metadata in {split}.")
            LOGGER.info(
                "Verified %s sample: image=%s, targets=%s",
                split,
                list(image.shape),
                list(target["boxes"].shape),
            )
pl.DataFrame(
    {
        "Split": ["Training stride 5", "Official validation"],
        "Frames": [selected.height, validation.height],
    }
)

## 3. Drive persistence and restore

We restore each experiment into its own local directory and verify the exact recipe.
Every callback writes only to that experiment's Drive folder. Saves are atomic,
snapshots are verified before publication, and there is one active writer per run.

In [ ]:
RUN_FOLDERS = {}
RESTORED_PROGRESS = {}
if SHOULD_TRAIN or SHOULD_EXPORT_SUBMISSIONS:
    from hm2.drive_api import (
        DriveApi,
        restore_checkpoint_from_drive_api,
        sync_checkpoint_to_drive_api,
        sync_results_to_drive_api,
    )

    api = DriveApi(service)
    batch_folder = api.folder(api.folder(drive_root, "batches"), BATCH_ID)
    cache = Path("/kaggle/temp/drive_cache") / BATCH_ID
    quota = service.about().get(fields="storageQuota").execute(num_retries=3)["storageQuota"]
    if "limit" in quota and int(quota["limit"]) - int(quota.get("usage", 0)) < 2 * 1024**3:
        raise OSError("Keep at least 2 GiB free on Drive for checkpoint replacement.")
    keys = selected["image_key"].to_list()
    selection_identity = compute_source_identity(IMAGE_ROOT, keys)
    for name in EXPERIMENTS:
        run_folder = api.folder(batch_folder, name)
        RUN_FOLDERS[name] = run_folder
        expected_config = build_experiment_config(
            SETTINGS_BY_EXPERIMENT[name], name, keys, selection_identity
        )
        restored = restore_checkpoint_from_drive_api(
            api, run_folder, OUTPUTS[name], expected_config, cache / name
        )
        RESTORED_PROGRESS[name] = restored
        LOGGER.info(
            "%s: restored progress %s",
            name,
            {
                key: restored.get(key)
                for key in ("epochs_completed", "batch_cursor", "global_step", "pending_evaluation")
            },
        )

## 4. Training, validation and submissions

The default is inference-only: we restore each completed final checkpoint, generate
predictions for all 3,811 test images and export the competition CSV. No training runs.
For future experiments set `SHOULD_TRAIN = True`; leave submission export enabled to
run test inference immediately after each completed experiment on the same Kaggle GPU.

CSV names include the experiment and stride-five selection. We retain the saved resize,
confidence threshold and NMS. Completed exports are reused only when checkpoint, test
metadata and CSV hashes agree. CSVs and metadata are committed to Drive snapshots and
copied to `/kaggle/working/submissions/` for download. No Kaggle submission is sent.
The session deadline covers the whole queue; incomplete training pauses before export.

In [ ]:
if SHOULD_TRAIN or SHOULD_EXPORT_SUBMISSIONS:
    deadline = time.monotonic() + SESSION_HOURS * 3600
    for name in EXPERIMENTS:
        settings = SETTINGS_BY_EXPERIMENT[name]
        output = OUTPUTS[name]
        progress = RESTORED_PROGRESS.get(name, {})
        is_completed = progress.get("epochs_completed", 0) == settings["training"][
            "epochs"
        ] and not progress.get("pending_evaluation", False)
        seconds = int(deadline - time.monotonic())
        if seconds <= 0:
            LOGGER.info("Session budget exhausted; rerun to continue from %s", name)
            break

        def persist_checkpoint(path: Path, folder_id: str = RUN_FOLDERS[name]) -> None:
            sync_checkpoint_to_drive_api(api, path, folder_id)

        if not is_completed:
            if not SHOULD_TRAIN:
                LOGGER.info("Skip incomplete %s: training is disabled", name)
                continue
            LOGGER.info("Start or resume %s with %s session seconds remaining", name, seconds)
            status = train_experiment(
                name,
                IMAGE_ROOT,
                METADATA,
                output,
                settings,
                seconds=seconds,
                on_checkpoint=persist_checkpoint,
            )
            display(status)
            publish_snapshot(output)
            persist_checkpoint(output)
            sync_results_to_drive_api(api, output, RUN_FOLDERS[name])
            if not status["is_completed"]:
                LOGGER.info("Paused %s; rerun this queue to resume", name)
                break
        else:
            LOGGER.info("Skip training for completed %s", name)
        if SHOULD_EXPORT_SUBMISSIONS:
            record = export_submission(name, IMAGE_ROOT, output, settings["evaluation"])
            publish_snapshot(output)
            persist_checkpoint(output)
            sync_results_to_drive_api(api, output, RUN_FOLDERS[name])
            SUBMISSIONS_ROOT.mkdir(parents=True, exist_ok=True)
            shutil.copy2(output / record["file"], SUBMISSIONS_ROOT / record["file"])
            display(record)
else:
    display({"Status": "Training and export disabled", "Experiments": EXPERIMENTS})

## 5. Saved measurements

We inspect final and best-validation metrics for each Faster R-CNN experiment.
Research diagnostics, curves and baseline comparisons remain in notebooks 05, 06
and 08. The completed YOLO11s results are recorded separately in notebook 07.

Downloaded CSV files are available under `/kaggle/working/submissions/`.

In [ ]:
display(metric_table({name: load_records(OUTPUTS[name]) for name in EXPERIMENTS}))